# 5 · Persistencia, aprobación humana y producción

**Objetivo:** que la memoria sobreviva a reinicios, que una persona **apruebe** lo que el agente
aprende y ver qué falta para llevar esto a producción.

**Empezar aquí sin haber hecho las secciones anteriores:** el agente que aprende de la sección 4 se
importa ya hecho con `build_learning_agent()`.

> Los ficheros se guardan en `.data/` en la raíz del repositorio. Si quieres empezar de cero, bórrala.

In [ ]:
import sqlite3
from datetime import datetime

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore
from langgraph.types import Command

from taller.agents import build_learning_agent, build_memory_store, build_retriever
from taller.config import ROOT, ruta_memoria
from taller.tools import Context
from taller.viz import mostrar_mensajes

retriever = build_retriever()
DATA = ROOT / ".data"  # carpeta de datos del taller, ignorada por git
DATA.mkdir(exist_ok=True)

## 5.1 · Persistir en disco

Cambiar `InMemorySaver`/`InMemoryStore` por sus versiones SQLite es **cambiar dos objetos**: el grafo
no se toca. En producción, lo mismo con `PostgresSaver`/`PostgresStore`.

In [ ]:
def abrir_agente():
    """Abre (o reabre) el agente sobre los mismos ficheros: simula reiniciar la aplicación."""
    checkpointer = SqliteSaver(sqlite3.connect(DATA / "taller.sqlite", check_same_thread=False))
    # ruta_memoria(): un fichero .sqlite por modelo de embeddings, para no mezclar vectores de distinto tamaño
    store_conn = sqlite3.connect(ruta_memoria(), check_same_thread=False, isolation_level=None)
    store = build_memory_store(SqliteStore, conn=store_conn)
    store.setup()  # crea las tablas la primera vez
    return build_learning_agent(retriever, checkpointer=checkpointer, store=store, require_approval=True)


agente = abrir_agente()
# Un hilo nuevo por ejecución del notebook: los ficheros persisten, y si reutilizáramos el hilo de una
# ejecución anterior el agente ya tendría el hecho aprendido y no volvería a pedir aprobación.
config = {"configurable": {"thread_id": f"persistente-{datetime.now():%H%M%S}"}}
ctx = Context(user_id="ana")

## 5.2 · Aprobación humana con `interrupt()`

Con `require_approval=True`, la herramienta `aprender_hecho` llama a `interrupt(...)` **antes** de guardar.
El grafo se detiene, guarda su estado en el checkpointer y devuelve el control: puede reanudarse segundos
o días después, incluso desde otro proceso. Tres pasos:

1. **Pausa**: el agente pide aprobación y la ejecución se para.
2. **Decisión**: una persona (tú, en la celda de abajo) responde `"si"` o `"no"`.
3. **Reanudación**: `Command(resume=...)` entrega la respuesta y la herramienta continúa.

Usamos `version="v2"`: `invoke` devuelve un `GraphOutput` con `.value` (el estado) e `.interrupts` (las pausas pendientes).

In [ ]:
resultado = agente.invoke(
    {"messages": [("human", "Apunta esto: el canal de incidencias ha pasado a llamarse #incidencias-cuac.")]},
    config,
    context=ctx,
    version="v2",
)

for intr in resultado.interrupts:
    print("⏸️  Pausado. El agente pide aprobación para guardar:")
    print("    ", intr.value["hecho"])
if not resultado.interrupts:
    print("Sin pausa: el modelo ha respondido sin llamar a aprender_hecho. Reformula («Guarda este cambio: ...»).")
    mostrar_mensajes(resultado.value["messages"][-1:])

**Paso 2, la decisión es tuya.** Cambia el valor y ejecuta las dos celdas siguientes; prueba las dos opciones.

In [ ]:
DECISION = "si"  # "si" guarda el hecho · "no" lo rechaza

**Paso 3, reanudar.** Antes, **simulamos un reinicio**: creamos el agente de nuevo desde los ficheros. La pausa
sigue ahí porque está en el checkpointer, no en memoria.

In [ ]:
agente = abrir_agente()  # «reinicio»: objetos nuevos, mismos ficheros
resultado = agente.invoke(Command(resume=DECISION), config, context=ctx, version="v2")
mostrar_mensajes(resultado.value["messages"][-2:])  # el resultado de la herramienta y la respuesta del agente

<details><summary><b>Cómo está hecho</b> (<code>taller/tools.py</code>)</summary>

```python
decision = interrupt(
    {"accion": "aprender_hecho", "hecho": hecho, "pregunta": "¿Apruebas guardar este conocimiento?"},
    # un objeto con un campo: Studio lo pinta como formulario con desplegable
    response_schema={"type": "object", "properties": {"decision": {"type": "string", "enum": ["si", "no"]}}},
)
if isinstance(decision, dict):  # respuesta del formulario de Studio
    decision = decision.get("decision")
if decision != "si":
    return "Una persona revisora ha rechazado guardar este hecho."
runtime.store.put(...)  # solo si se aprueba
```

Tres reglas de `interrupt()`: al reanudar, la herramienta **se vuelve a ejecutar desde el principio** y
`interrupt()` devuelve el valor (por eso el `put` va después); no envolverlo en `try/except` (nunca pausaría);
y `Command(resume=...)` solo tiene sentido con una pausa pendiente (si no, el agente responde sin más).
</details>

## 5.3 · Inspeccionar y viajar en el tiempo

El checkpointer guarda **cada paso**. Podemos ver el estado actual y todo el historial: es la base para
depurar, auditar y «viajar en el tiempo» (reanudar desde un punto anterior con otro camino).

In [ ]:
actual = agente.get_state(config)
print("Mensajes en el hilo:", len(actual.values["messages"]))
print("Siguiente nodo:", actual.next or "(ninguno: terminado)")

for snapshot in agente.get_state_history(config):
    ultimo = snapshot.values["messages"][-1] if snapshot.values.get("messages") else None
    print(
        f"paso {snapshot.metadata.get('step'):>2} · siguiente={snapshot.next} · último={type(ultimo).__name__}"
    )

Para reanudar desde un punto anterior basta con pasar su `checkpoint_id` en la configuración:

```python
anterior = list(agente.get_state_history(config))[3]
agente.invoke(None, anterior.config, context=ctx)  # vuelve a ejecutar desde ese punto
```

## 5.4 · Robustez por nodo

Cada nodo puede declarar su propia política de reintentos y un tiempo máximo (novedad de LangGraph 1.2):

```python
from langgraph.types import RetryPolicy

builder.add_node("agent", agent, retry_policy=RetryPolicy(max_attempts=3), timeout=30)
```

Y al ejecutar, `durability` decide cuándo se escribe cada checkpoint: `"exit"` (al terminar, más rápido),
`"async"` (por defecto) o `"sync"` (antes de cada paso, lo más seguro).


## 5.5 · El mismo grafo, servido: `langgraph.json` y el Agent Server

Lo que hemos hecho a mano en este notebook (checkpointer, store, pausar y reanudar) es justo lo que el **Agent
Server** hace por nosotros. `langgraph.json` declara los grafos; `uv run studio` los sirve en local con
persistencia en memoria, y el mismo fichero sirve para desplegarlos (en LangSmith Deployment o en tu propia
infraestructura con `langgraph build`, que genera una imagen Docker que usa Postgres y Redis).

Fíjate en que los grafos de `taller/studio.py` se compilan **sin** checkpointer ni store: los pone el servidor y
los inyecta en `runtime.store`. La búsqueda semántica del store también se configura ahí (`store.index`).

In [ ]:
print((ROOT / "langgraph.json").read_text())

La aprobación humana funciona igual por HTTP: la ejecución devuelve `__interrupt__` y se reanuda con un
`command={"resume": ...}`. En Studio, la ejecución se queda en pausa y aparece un cuadro para responder.

In [ ]:
from taller.servidor import cliente, contenido

client = cliente()
if client:
    hilo = client.threads.create()["thread_id"]
    r = client.runs.wait(
        hilo,
        "agente_que_aprende",
        input={
            "messages": [{"role": "human", "content": "Apunta esto: la demo general pasa a los viernes."}]
        },
        context={"user_id": "ana"},
    )
    if r.get("__interrupt__"):
        print("Pendiente de aprobación:", r["__interrupt__"][0]["value"]["hecho"])
        r = client.runs.wait(hilo, "agente_que_aprende", command={"resume": "si"}, context={"user_id": "ana"})
        print("Tras aprobar:", contenido(r["messages"][-1])[:120])
    else:  # el modelo no ha llamado a aprender_hecho: pídeselo de forma más explícita
        print("Sin interrupción:", contenido(r["messages"][-1])[:120])

## 5.6 · Observabilidad: ver qué ha pasado de verdad

Un agente en producción falla de formas que no se ven en un `print`: la herramienta equivocada, un prompt que
creció sin control, 40 segundos de latencia en un nodo. Las **trazas** guardan cada ejecución completa: prompts,
llamadas a herramientas, tokens, tiempos y errores, con el árbol de nodos del grafo. Dos opciones:

| | **LangSmith** | **Langfuse** |
|---|---|---|
| Quién | LangChain (la misma cuenta que Studio) | Código abierto (MIT), se puede autoalojar |
| Activar | `LANGSMITH_TRACING=true` + `LANGSMITH_API_KEY` en `.env`; sin tocar código | `LANGFUSE_PUBLIC_KEY` + `LANGFUSE_SECRET_KEY` y un *callback* en cada llamada |
| Gratis | 5 000 trazas/mes en el plan Developer | 50 000 unidades/mes en el plan Hobby, o ilimitado si lo alojas tú |
| Dónde | Nube de LangSmith (EE. UU. o UE) | Nube (UE, EE. UU.) o tu servidor (`docker compose`) |

`taller.observabilidad` lo encapsula: `con_trazas(config)` añade el *callback* de Langfuse si está configurado, y
LangSmith no necesita nada. En el servidor de `langgraph dev` solo funciona LangSmith (el grafo se exporta compilado,
sin *callbacks* propios). Si no tienes ninguna cuenta, esta celda simplemente te lo dice.

In [ ]:
from taller.observabilidad import con_trazas, estado

print(estado())  # qué plataformas de trazas hay activas

agente = abrir_agente()
resultado = agente.invoke(
    {"messages": [("human", "¿Qué significa el error ERR-4012?")]},
    con_trazas({"configurable": {"thread_id": "trazas-1"}}),
    context=ctx,
)
print(resultado["messages"][-1].text[:200])

## 5.7 · De aquí a producción

| Necesidad | Pieza |
|---|---|
| Persistencia real y concurrente | `PostgresSaver` + `PostgresStore` (con pgvector para la búsqueda semántica) |
| Saber qué hace el agente | Trazas con LangSmith o Langfuse (5.6), u OpenTelemetry |
| Saber si lo hace **bien** | Evaluaciones: un conjunto de preguntas con respuestas esperadas, en CI |
| Servirlo como API | El Agent Server con `langgraph.json` (5.5), o tu propio FastAPI con `graph.stream(...)` |
| Streaming hacia la interfaz | `stream(..., version="v2")` o el nuevo `stream_events(version="v3")` (beta) |
| Conversaciones largas | Resumir o recortar el historial (p. ej. `SummarizationMiddleware` en `create_agent`) |
| Calidad del RAG | Mejores fragmentos, filtros por metadatos, *reranking*, evaluaciones de recuperación |
| Memoria sana | Aprobación humana, TTL, deduplicación y revisión periódica de lo aprendido |

> **Para recordar:** el grafo no cambia entre el notebook, Studio y producción. Cambian las piezas que le
> enchufas (checkpointer, store, trazas) y quién lo invoca (tu código, el Agent Server, una interfaz).

### 🚀 Reto opcional

Ejecuta la demo de terminal, que usa este mismo agente con persistencia y aprobación humana:

```bash
uv run python -m taller.demo
```
